# 23 — Quantum State Preparation

## Purpose

Calibrate and demonstrate multiple cavity state preparation methods: SNAP gates, Echoed Conditional Displacement (ECD), GRAPE optimal control, selective number-dependent rotations, displaced Fock states, and Schrödinger cat states. Each method is verified via Wigner tomography.

## Methodology

1. **SNAP calibration** — sweep SNAP gate parameters to generate calibration data, then optimize the gate angles
2. **SNAP Wigner verification** — measure the Wigner function of the SNAP-prepared state
3. **ECD state preparation** — prepare a target cavity state using echoed conditional displacements, then verify via Wigner
4. **GRAPE state preparation** — apply a numerically-optimized (GRAPE) control pulse and verify via Wigner
5. **Selective number-dependent rotation** — SNAP + displacement combination for Fock-selective rotations
6. **Displaced Fock state** — prepare |α, n⟩ and verify via Wigner
7. **Cat state** — prepare |α⟩ + |−α⟩ and verify via Wigner

## Expected Outcomes

- SNAP gate fidelity sufficient for deterministic Fock-state preparation
- Wigner functions matching expected theoretical distributions for each prepared state
- Cat state showing characteristic interference fringes in the Wigner function

## Prerequisites

- **Notebook 08** — displacement pulse waveforms defined
- **Notebook 21** — storage cavity characterized (frequency, χ)
- **Notebook 22** — Fock-resolved dynamics characterized (validates Hamiltonian model)

## 1. Setup — Session Bootstrap

Open the notebook stage and load the Fock-resolved experiments checkpoint from notebook 22.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    SNAPOptimization,
    StorageWignerTomography,
    fit_quality_gate,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="23_quantum_state_preparation",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

fock_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="22_fock_resolved_experiments",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if fock_checkpoint is not None:
    print(f"Prior stage 22 status: {fock_checkpoint['status']}")

## 2. Configuration — State Preparation Defaults

Set SNAP parameters, Wigner grid resolution, ECD averaging, and general state-preparation averaging counts.

In [ ]:
APPLY_SNAP_CALIBRATION = True

# --- SNAP (Exp 68, 69) ---
SNAP_MAX_N = 5
SNAP_N_AVG = 10000

# --- Wigner verification (Exp 70, 72, 74) ---
WIGNER_ALPHA_MAX = 3.0
WIGNER_ALPHA_STEP = 0.1
WIGNER_N_AVG = 10000

# --- ECD (Exp 71) ---
ECD_N_AVG = 10000

# --- General state prep ---
STATE_PREP_N_AVG = 10000

print("State preparation settings:")
print(f"  SNAP max_n: {SNAP_MAX_N}, n_avg: {SNAP_N_AVG}")
print(f"  Wigner: α_max={WIGNER_ALPHA_MAX}, dα={WIGNER_ALPHA_STEP}")
print(f"  ECD n_avg: {ECD_N_AVG}")
print(f"  General state prep n_avg: {STATE_PREP_N_AVG}")

## 3. Execution — SNAP Calibration Data Generation

Sweep SNAP gate parameters (phase angles per Fock level) to produce the calibration dataset needed by the optimizer.

In [ ]:
snap_data_result = session.snap_calibration_sweep(
    max_n=SNAP_MAX_N,
    n_avg=SNAP_N_AVG,
)

print("SNAP calibration data generation complete.")

## 4. Execution — SNAP Gate Optimization

Optimize the SNAP gate angles using the calibration data. The optimized angles are applied to the session if the fit passes the quality gate.

In [ ]:
snap_opt = SNAPOptimization(session)
snap_opt_result = snap_opt.run(n_avg=SNAP_N_AVG)
snap_opt_analysis = snap_opt.analyze(snap_opt_result, update_calibration=True)
snap_opt.plot(snap_opt_analysis)

snap_fit_ok = fit_quality_gate(snap_opt_analysis, r_squared_min=0.80)

if snap_fit_ok:
    snap_patch, _, snap_apply = preview_or_apply_patch_ops(
        session,
        reason="SNAP gate optimization",
        proposed_patch_ops=snap_opt_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_SNAP_CALIBRATION,
    )
    if snap_apply is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
else:
    print("WARNING: SNAP optimization fit quality gate FAILED — skipping apply.")

print("SNAP optimization complete.")

## 5. Verification — SNAP Wigner Tomography

Measure the Wigner function of the SNAP-prepared state to verify that the optimized gate angles produce the target Fock-state distribution.

In [ ]:
wigner_snap = StorageWignerTomography(session)
wigner_snap_result = wigner_snap.run(
    alpha_max=WIGNER_ALPHA_MAX,
    alpha_step=WIGNER_ALPHA_STEP,
    n_avg=WIGNER_N_AVG,
    state_prep="snap",
)
wigner_snap_analysis = wigner_snap.analyze(wigner_snap_result, update_calibration=False)
wigner_snap.plot(wigner_snap_analysis)

print("SNAP Wigner verification complete.")

## 6. Execution — ECD State Preparation

Prepare a target cavity state using the Echoed Conditional Displacement (ECD) protocol. ECD interleaves conditional displacements with qubit echo pulses to build up arbitrary cavity states.

In [ ]:
ecd_result = session.ecd_state_preparation(
    n_avg=ECD_N_AVG,
)

print("ECD state preparation complete.")

## 7. Verification — ECD Wigner Tomography

Measure the Wigner function of the ECD-prepared state to verify fidelity to the target.

In [ ]:
wigner_ecd = StorageWignerTomography(session)
wigner_ecd_result = wigner_ecd.run(
    alpha_max=WIGNER_ALPHA_MAX,
    alpha_step=WIGNER_ALPHA_STEP,
    n_avg=WIGNER_N_AVG,
    state_prep="ecd",
)
wigner_ecd_analysis = wigner_ecd.analyze(wigner_ecd_result, update_calibration=False)
wigner_ecd.plot(wigner_ecd_analysis)

print("ECD Wigner verification complete.")

## 8. Execution — GRAPE Optimal Control State Preparation

Apply a numerically-optimized control pulse (GRAPE — Gradient Ascent Pulse Engineering) to prepare a target cavity state in a single shot.

In [ ]:
grape_result = session.grape_state_preparation(
    n_avg=STATE_PREP_N_AVG,
)

print("GRAPE state preparation complete.")

## 9. Verification — GRAPE Wigner Tomography

Measure the Wigner function of the GRAPE-prepared state to verify fidelity to the target.

In [ ]:
wigner_grape = StorageWignerTomography(session)
wigner_grape_result = wigner_grape.run(
    alpha_max=WIGNER_ALPHA_MAX,
    alpha_step=WIGNER_ALPHA_STEP,
    n_avg=WIGNER_N_AVG,
    state_prep="grape",
)
wigner_grape_analysis = wigner_grape.analyze(wigner_grape_result, update_calibration=False)
wigner_grape.plot(wigner_grape_analysis)

print("GRAPE Wigner verification complete.")

## 10. Execution — Selective Number-Dependent Rotation

Apply a SNAP + displacement combination to perform a rotation conditioned on a specific cavity Fock state, enabling Fock-selective quantum operations.

In [ ]:
sndr_result = session.selective_number_dependent_rotation(
    n_avg=STATE_PREP_N_AVG,
)

print("Selective number-dependent rotation complete.")

## 11. Execution — Displaced Fock State |α, n⟩

Prepare a displaced Fock state by first creating a Fock state |n⟩ then displacing by α. Verified via Wigner tomography.

In [ ]:
disp_fock_result = session.displaced_fock_state_preparation(
    n_avg=STATE_PREP_N_AVG,
)

wigner_dfock = StorageWignerTomography(session)
wigner_dfock_result = wigner_dfock.run(
    alpha_max=WIGNER_ALPHA_MAX,
    alpha_step=WIGNER_ALPHA_STEP,
    n_avg=WIGNER_N_AVG,
    state_prep="displaced_fock",
)
wigner_dfock_analysis = wigner_dfock.analyze(wigner_dfock_result, update_calibration=False)
wigner_dfock.plot(wigner_dfock_analysis)

print("Displaced Fock state preparation + Wigner complete.")

## 12. Execution — Schrödinger Cat State

Prepare the superposition |α⟩ + |−α⟩ (even cat state). The Wigner function should show two Gaussian lobes with quantum interference fringes between them.

In [ ]:
cat_result = session.cat_state_preparation(
    n_avg=STATE_PREP_N_AVG,
)

wigner_cat = StorageWignerTomography(session)
wigner_cat_result = wigner_cat.run(
    alpha_max=WIGNER_ALPHA_MAX,
    alpha_step=WIGNER_ALPHA_STEP,
    n_avg=WIGNER_N_AVG,
    state_prep="cat",
)
wigner_cat_analysis = wigner_cat.analyze(wigner_cat_result, update_calibration=False)
wigner_cat.plot(wigner_cat_analysis)

print("Cat state preparation + Wigner complete.")

## 13. Checkpoint — Save State Preparation Results

Persist the SNAP calibration and characterization metrics. Only SNAP angles are applied as a calibration; all other methods are verification-only.

In [ ]:
snap_applied = "snap_apply" in dir() and snap_apply is not None

stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="23_quantum_state_preparation",
    status="calibrated" if snap_applied else "characterized",
    summary="SNAP optimization, ECD, GRAPE, displaced Fock, and cat state preparation with Wigner verification.",
    consumed_inputs={
        "snap_max_n": SNAP_MAX_N,
        "snap_n_avg": SNAP_N_AVG,
        "wigner_alpha_max": WIGNER_ALPHA_MAX,
        "wigner_alpha_step": WIGNER_ALPHA_STEP,
        "ecd_n_avg": ECD_N_AVG,
        "state_prep_n_avg": STATE_PREP_N_AVG,
    },
    persisted_outputs={
        "snap_applied": snap_applied,
    },
    advisory_outputs={},
    next_stage="24_free_evolution_tomography",
    notes=[
        "SNAP calibration is the only applied calibration; all else is characterization.",
        "ECD, GRAPE, and direct state prep methods depend on correct displacement ops (notebook 08).",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")